In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
from sys import path
path.insert(0, '..')
from scripts.logistic_reg import logistic_regression_loss,get_L_one_hot,train_log_reg
from scripts.utility import vCol,map_labels_into_numbers
import scipy
import json
import pandas as pd

In [2]:
D=np.load('../data/processed/D_train_latent.npy')
L=np.load('../data/processed/L_train.npy')
labels_to_num=json.load(open('../data/processed/labels_to_num.json','r'))
num_to_labels=json.load(open('../data/processed/num_to_label.json','r'))
D_test=np.load('../data/processed/D_test.npy')
L_test=np.load('../data/processed/L_test.npy')

In [5]:
D

array([[ 4.90444361,  3.62614966,  3.72809099, ..., -7.01448752,
        -6.84106682, -6.48973027],
       [-4.63251198, -5.74676351, -5.74217453, ...,  1.09870867,
         0.99162572,  0.75265794],
       [-0.92876343, -0.1555162 , -0.43350918, ...,  0.45251673,
         0.52603201,  0.40703671],
       [ 0.16578367,  0.19910355,  0.12028995, ...,  2.08591049,
         2.45074201,  2.88660849],
       [ 0.96125541, -1.1642225 ,  0.28296175, ..., -0.34692774,
        -0.59530938, -0.55492143]], shape=(5, 7352))

In [6]:
lambdas=[1e-4, 1e-2, 0.1, 1.0]
best_w=0
best_b=0
best_acc=0
for l in lambdas:
    w,b=train_log_reg(D,L,l)
    s=w.T @ D_test + b
    predictions=np.argmax(s,axis=0)
    accuracy=np.mean(predictions==L_test)
    if accuracy > best_acc:
        best_acc=accuracy
        best_w=w
        best_b=b
    print(f"Accuracy with lambda= {l} is: {accuracy*100}%")

Accuracy with lambda= 0.0001 is: 86.01968103155751%
Accuracy with lambda= 0.01 is: 87.20732948761453%
Accuracy with lambda= 0.1 is: 87.10553104852391%
Accuracy with lambda= 1.0 is: 84.8320325755005%


In [7]:
import numpy as np

def export_weights_to_rust(W, b, filename="../../ESP32-Rust/src/model_weights.rs"):
    with open(filename, "w") as f:
        f.write("// File autogenerato: Parametri della Regressione Logistica\n")
        f.write("#![allow(dead_code)]\n\n")
        
        # Esporta la matrice dei pesi W (5x6) appiattita in un array 1D
        w_flat = W.flatten(order='C')
        f.write(f"pub static W_MATRIX: [f32; {len(w_flat)}] = [\n    ")
        w_str = ", ".join([f"{val:.6f}" for val in w_flat])
        f.write(w_str)
        f.write("\n];\n\n")
        
        # Esporta il vettore dei bias b (6x1)
        b_flat = b.flatten()
        f.write(f"pub static B_VECTOR: [f32; {len(b_flat)}] = [\n    ")
        b_str = ", ".join([f"{val:.6f}" for val in b_flat])
        f.write(b_str)
        f.write("\n];\n")
        
    print(f"Pesi esportati con successo in {filename}")

# Richiamo della funzione passando i parametri ottimali calcolati
export_weights_to_rust(best_w, best_b)

Pesi esportati con successo in ../../ESP32-Rust/src/model_weights.rs


In [8]:
import numpy as np
import os

def export_test_data_to_rust(X_test, y_test, num_samples=50, filename="../../ESP32-Rust/src/test_data.rs"):
    # Assicura che la directory di destinazione esista
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    
    # Estrazione del sottoinsieme di test
    X_sub = X_test[:num_samples]
    y_sub = y_test[:num_samples]
    
    with open(filename, "w") as f:
        f.write("// File autogenerato: Dataset di validazione per ESP32\n")
        f.write("#![allow(dead_code)]\n\n")
        
        # Definizione della costante globale per il numero di campioni
        f.write(f"pub const NUM_SAMPLES: usize = {num_samples};\n\n")
        
        # Generazione della matrice delle feature (array 2D in Rust)
        f.write(f"pub static TEST_FEATURES: [[f32; 5]; {num_samples}] = [\n")
        for row in X_sub:
            # Formattazione lineare di ogni riga a 6 cifre decimali
            row_str = ", ".join([f"{val:.6f}" for val in row])
            f.write(f"    [{row_str}],\n")
        f.write("];\n\n")
        
        # Generazione del vettore delle etichette reali
        f.write(f"pub static TEST_LABELS: [usize; {num_samples}] = [\n")
        labels_str = ", ".join([str(int(val)) for val in y_sub])
        f.write(f"    {labels_str}\n")
        f.write("];\n")
        
    percorso_assoluto = os.path.abspath(filename)
    print(f"Dataset di validazione ({num_samples} campioni) esportato con successo in:\n{percorso_assoluto}")

# LA CORREZIONE È QUI: Trasponiamo D_test con .T prima di elaborarla
export_test_data_to_rust(D_test.T, L_test)

Dataset di validazione (50 campioni) esportato con successo in:
/home/emmanuelmessina00/Scrivania/edge-biomechanics-classifier/ESP32-Rust/src/test_data.rs
